Imports


In [1]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

e:\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.2.0
e:\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

e:\Regression_ML_EndtoEnd\.venv\Lib\site-packages\mlflow\utils\autologging_utils\versioning.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [5]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("../data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("../data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [10]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import mlflow

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "random_state": 42,
        "n_jobs": 1
    }

    with mlflow.start_run(nested=True):

        mlflow.log_params(params)

        model = XGBRegressor(**params)

        model.fit(X_train, y_train)

        preds = model.predict(X_eval)

        mae = mean_absolute_error(y_eval, preds)

        mlflow.log_metric("mae", mae)

    return mae

In [11]:
# ==============================================
# Run Optuna study with MLflow
# ==============================================

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=5,
    catch=(Exception,)
)

print("Best params:")
print(study.best_trial.params)

[I 2026-06-15 18:43:58,093] A new study created in memory with name: no-name-96726fd0-b681-4e14-91df-ea1d14aa4a9f
[I 2026-06-15 18:44:57,566] Trial 0 finished with value: 37739.64400721333 and parameters: {'n_estimators': 165, 'max_depth': 3, 'learning_rate': 0.2504851464623158, 'subsample': 0.8143955400311403, 'colsample_bytree': 0.8802991240928364}. Best is trial 0 with value: 37739.64400721333.
[I 2026-06-15 18:45:23,154] Trial 1 finished with value: 49814.37858245586 and parameters: {'n_estimators': 101, 'max_depth': 4, 'learning_rate': 0.02329557049945882, 'subsample': 0.7084305384296244, 'colsample_bytree': 0.738221362785418}. Best is trial 0 with value: 37739.64400721333.
[I 2026-06-15 18:46:27,153] Trial 2 finished with value: 34024.6746391404 and parameters: {'n_estimators': 394, 'max_depth': 5, 'learning_rate': 0.29765045196482237, 'subsample': 0.9090928530900155, 'colsample_bytree': 0.7804158544917086}. Best is trial 2 with value: 34024.6746391404.
[I 2026-06-15 18:47:10,668

Best params:
{'n_estimators': 296, 'max_depth': 5, 'learning_rate': 0.13023543418550806, 'subsample': 0.8532752859291528, 'colsample_bytree': 0.8436004349759152}


In [13]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, artifact_path="model")

Final tuned model performance:
MAE: 33893.44664327502
RMSE: 75353.53613086796
R²: 0.9561199305885101


e:\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [18:49:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2026/06/15 18:50:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [14]:
# Save a local copy
import joblib
joblib.dump(best_model, "../models/xgboost_best_model.pkl")

print("✅ Model saved locally and logged to MLflow.")

✅ Model saved locally and logged to MLflow.
